In [1]:
import copy
from importlib import simple
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from bs4.element import XMLProcessingInstruction
from torch import Tensor
from torch.nn.functional import channel_shuffle

print("PyTorch version", torch.__version__)

PyTorch version 2.13.0+cpu


In [2]:
"""
为什么需要控制中间激活的尺度
    在每一层接受激活后后,先根据当前batch的均值和方差进行标准化,再交给后续计算
"""
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
mean = x.mean()
var = x.var(correction=0)

x_hat = (x - mean) / (var + 1e-8).sqrt()
x_hat_mean = x_hat.mean()
x_hat_var = x_hat.var(correction=0)

print("Input values:", x)
print("Mean:", mean.item())
print("Variance:", var.item())
print(x_hat)
print(x_hat_mean.item())
print(x_hat_var.item())


Input values: tensor([1., 2., 3., 4.])
Mean: 2.5
Variance: 1.25
tensor([-1.3416, -0.4472,  0.4472,  1.3416])
0.0
0.9999998807907104


In [3]:
"""
BatchNorm 在对哪些维度归一化

    保留特征维度C,在其余属于同一特征的样本维度上计算
"""
x = torch.tensor(
    [
        [1.0, 10.0, 100.0],
        [2.0, 20.0, 200.0],
        [3.0, 30.0, 300.0],
        [4.0, 40.0, 400.0]
    ]
)

feature_mean = x.mean(dim=0)
feature_var = x.var(dim=0, correction=0)
x_hat = (x - feature_mean) / (feature_var + 1e-8).sqrt()

print("Feature mean:", feature_mean)
print("Feature variance:", feature_var)
print("Normalized tensor:", x_hat)
print('Mean of each  feature:', x_hat.mean(dim=0))
print('Variance of each  feature:', x_hat.var(dim=0, correction=0))

Feature mean: tensor([  2.5000,  25.0000, 250.0000])
Feature variance: tensor([1.2500e+00, 1.2500e+02, 1.2500e+04])
Normalized tensor: tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]])
Mean of each  feature: tensor([0.0000e+00, 2.9802e-08, 2.9802e-08])
Variance of each  feature: tensor([1.0000, 1.0000, 1.0000])


In [4]:
x = torch.arange(2 * 3 * 2 * 2, dtype=torch.float)
x = x.reshape(2, 3, 2, 2)

dim = (0, 2, 3)
channel_mean = x.mean(dim, keepdim=True)
channel_var = x.var(dim, keepdim=True)
x_hat = (x - channel_mean) / (channel_var + 1e-8).sqrt()

print('Input shape:', x.shape)
print('Channel mean shape:', channel_mean.shape)
print('Channel means:', channel_mean.flatten())
print('Channel variances:', channel_var.flatten())

print("Normalized channel means:", x_hat.mean(dim))
print("Normalized channel variances:", x_hat.var(dim, correction=0))


Input shape: torch.Size([2, 3, 2, 2])
Channel mean shape: torch.Size([1, 3, 1, 1])
Channel means: tensor([ 7.5000, 11.5000, 15.5000])
Channel variances: tensor([42.5714, 42.5714, 42.5714])
Normalized channel means: tensor([-2.9802e-08, -2.9802e-08, -2.9802e-08])
Normalized channel variances: tensor([0.8750, 0.8750, 0.8750])


In [5]:
"""
为什么标准化以后需要gamma和beta
    batchNorm 总是强制输出均值为0,方差为1,它可能会限制网络的表达能力,为了解决这个问题,在BatchNorm在标准化后加入这两个参数

"""
batch_norm = nn.BatchNorm1d(4)
print("gamma:", batch_norm.weight)
print("beta:", batch_norm.bias)

gamma: Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)
beta: Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)


In [6]:
"""
训练阶段的统计量计算与batch依赖
"""
batch_norm = nn.BatchNorm1d(2, affine=False, track_running_stats=False)
batch_norm.train()

sample = torch.tensor([[1.0, 2.0]])
other_sample1 = torch.tensor([[2.0, 4.0], [3.0, 6.0], [4.0, 8.0]])
other_sample2 = torch.tensor([[10.0, 20.0], [20.0, 40.0], [30.0, 60.0]])

batch1 = torch.concat([sample, other_sample1], dim=0)
batch2 = torch.concat([sample, other_sample2], dim=0)

output1 = batch_norm(batch1)[0]
output2 = batch_norm(batch2)[0]

print("same simple in batch A :", output1)
print("same simple in batch B :", output2)


same simple in batch A : tensor([-1.3416, -1.3416])
same simple in batch B : tensor([-1.3136, -1.3136])


In [7]:

"""
推理阶段为什么不能继续依赖当前的batch
    - 单样本或者小batch的均值,方差不可信
    - 同一个样本的徐策或收到同样的样本影响
    推理阶段要加上mode.eval()
"""
x = torch.tensor(
    [
        [1.0, 10.0, 100.0],
        [2.0, 20.0, 200.0],
        [3.0, 30.0, 300.0],
        [4.0, 40.0, 400.0]
    ]
)

batch_norm = nn.BatchNorm1d(3)
batch_norm.train()
for _ in range(10):
    y = batch_norm(x + torch.randn_like(x))
print('Running mean:', batch_norm.running_mean)
print('Running variance:', batch_norm.running_var)

batch_norm.eval()
with torch.no_grad():
    y_eval = batch_norm(x)

print('Training output mean:', y.mean(dim=0))
print('Evaluation output mean:', y_eval.mean(dim=0))


Running mean: tensor([  1.8271,  16.3084, 162.8789])
Running variance: tensor([2.2231e+00, 1.0899e+02, 1.0855e+04])
Training output mean: tensor([ 2.9802e-08,  1.1921e-07, -2.0862e-07], grad_fn=<MeanBackward1>)
Evaluation output mean: tensor([0.4513, 0.8326, 0.8362])


In [8]:
"""
Running Statistics 和 Momentum
"""

batch_norm = nn.BatchNorm1d(2, momentum=0.1)
batch_norm.train()

x1 = torch.tensor([[0.0, 10.0], [2.0, 14.0]])
x2 = torch.tensor([[10.0, 20.0], [14.0, 28.0]])
print("Initial  running mean:", batch_norm.running_mean)

y1 = batch_norm(x1)
print("After batch 1:", batch_norm.running_mean)
y2 = batch_norm(x2)
print("After batch 2:", batch_norm.running_mean)

Initial  running mean: tensor([0., 0.])
After batch 1: tensor([0.1000, 1.2000])
After batch 2: tensor([1.2900, 3.4800])


In [9]:
"""
    BatchNorm的PyTorch实现
     - 可学习参数weight和bias
     - Buffer Running_mean  和 running_var
     - 训练阶段的running  statistics
     - 对running variance的无偏修正
"""


def batch_norm(
        x: Tensor,
        running_mean: Tensor,
        running_var: Tensor,
        weight: Tensor | None = None,
        bias: Tensor | None = None,
        training: bool = False,
        momentum: float = 0.1,
        eps: float = 1e-5,

) -> Tensor:
    if x.ndim < 2:
        raise AssertionError("")
    reduce_dims = (0, *range(2, x.ndim))
    broadcast_shape = (1, x.size(1)) + (1,) * (x.ndim - 2)
    if training:
        sample_count = x.numel() // x.size(1)
        if sample_count <= 1:
            raise AssertionError("")
        batch_mean = x.mean(dim=reduce_dims)
        batch_var = x.var(dim=reduce_dims, correction=0)
        unbiased_var = batch_var * sample_count / (sample_count - 1)

        with torch.no_grad():
            running_mean.lerp_(batch_mean, momentum)
            running_var.lerp_(unbiased_var, momentum)
    else:
        batch_mean = running_mean
        batch_var = running_var

    batch_mean = batch_mean.reshape(broadcast_shape)
    batch_var = batch_var.reshape(broadcast_shape)

    y = (x - batch_mean) / (batch_var + eps).sqrt()
    if weight is not None:
        y = y * weight.reshape(broadcast_shape)
    if bias is not None:
        y = y + bias.reshape(broadcast_shape)
    return y


x = torch.randn(16, 3, 32, 32)
weight = torch.randn(3)

bias = torch.randn(3)
running_mean = torch.zeros(3)
running_var = torch.ones(3)

actual = batch_norm(x,
                    running_mean=running_mean,
                    running_var=running_var,
                    weight=weight,
                    bias=bias)
expected = F.batch_norm(x, running_mean=running_mean, running_var=running_var, weight=weight, bias=bias)

max_diff = (actual - expected).abs().max()
print("Maximum difference:", max_diff.item())


Maximum difference: 1.1920928955078125e-07


In [10]:
class BatchNorm(nn.Module):
    weight: Tensor | None
    bias: Tensor | None
    running_mean: Tensor
    running_var: Tensor

    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1, affine: bool = True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        self.affine = affine

        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)

        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x: Tensor) -> Tensor:
        if x.size(1) != self.num_features:
            raise AssertionError(f"Excepted {self.num_features} channels,but got {x.size(1)} channels")
        return batch_norm(
            x,
            self.running_mean,
            self.running_var,
            weight=self.weight,
            bias=self.bias,
            training=self.training,
            momentum=self.momentum,
            eps=self.eps,
        )

    def extra_repr(self) -> str:
        return (
            f"num_features={self.num_features}, eps={self.eps}, momentum={self.momentum}, affine={self.affine}"
        )


x = torch.randn(8, 4)
batch_norm1 = BatchNorm(4)
batch_norm2 = nn.BatchNorm1d(4)

with  torch.no_grad():
    batch_norm2.weight.copy_(batch_norm1.weight)
    batch_norm2.bias.copy_(batch_norm1.bias)

batch_norm1.train()
batch_norm2.train()

actual = batch_norm1(x)
expected = batch_norm2(x)

max_diff = (actual - expected).abs().max()
diff_mean = (batch_norm1.running_mean - batch_norm2.running_mean).abs().max()
diff_var = (batch_norm1.running_var - batch_norm2.running_var).abs().max()

print("Maximum training  difference:", max_diff.item())
print("Running mean difference:", diff_mean.item())
print("Running variance difference:", diff_var.item())


Maximum training  difference: 2.384185791015625e-07
Running mean difference: 0.0
Running variance difference: 5.960464477539063e-08


In [12]:
"""
BatchNor1d,BatchNor2d,BatchNor3d
"""

x_1d = torch.randn(4, 8, 16)
x_2d = torch.randn(4, 8, 16, 16)
x_3d = torch.randn(4, 8, 4, 16, 16)

batch_norm_1d = nn.BatchNorm1d(8)
batch_norm_2d = nn.BatchNorm2d(8)
batch_norm_3d = nn.BatchNorm3d(8)

print('BatchNor1d output:', batch_norm_1d(x_1d).shape)
print('BatchNor2d output:', batch_norm_2d(x_2d).shape)
print('BatchNor3d output:', batch_norm_3d(x_3d).shape)




BatchNor1d output: torch.Size([4, 8, 16])
BatchNor2d output: torch.Size([4, 8, 16, 16])
BatchNor3d output: torch.Size([4, 8, 4, 16, 16])


In [16]:
"""
Conv-BN Fusion 的PyTorch 实现
"""


def fuse_conv_bn_eval(
        conv: nn.Conv2d,
        bn: nn.BatchNorm2d,
) -> nn.Conv2d:
    if conv.training or bn.training:
        raise AssertionError('Both conv bn must be in eval mode')
    if conv.out_channels != bn.num_features:
        raise AssertionError('conv.out_channels != bn.num_features')
    fused_conv = copy.deepcopy(conv)

    conv_weight = conv.weight
    if conv.bias is None:
        conv_bias = torch.zeros(
            conv.out_channels,
            device=conv_weight.device,
            dtype=conv_weight.dtype,
        )
    else:
        conv_bias = conv.bias

    if bn.affine:
        gamma = bn.weight
        beta = bn.bias
    else:
        gamma = torch.ones_like(bn.running_mean)
        beta = torch.zeros(bn.running_mean)
    scale = gamma * (bn.running_var + bn.eps).rsqrt()
    fused_weight = conv_weight * scale.reshape(-1, 1, 1, 1)
    fused_bias = (conv_bias - bn.running_mean) * scale + beta

    fused_conv.weight = nn.Parameter(fused_weight)
    fused_conv.bias = nn.Parameter(fused_bias)
    return fused_conv


conv = nn.Conv2d(
    in_channels=3,
    out_channels=8,
    kernel_size=3,
    padding=1,
    bias=False,
)

batch_norm = nn.BatchNorm2d(8)

conv.train()
batch_norm.train()

for _ in range(20):
    x = torch.randn(16, 3, 16, 16)
    y = batch_norm(conv(x))

conv.eval()
batch_norm.eval()

fused_conv = fuse_conv_bn_eval(conv, batch_norm)
fused_conv.eval()

x = torch.randn(4, 3, 16, 16)
with   torch.inference_mode():
    output = batch_norm(conv(x))
    fused_output = fused_conv(x)
max_diff = (output - fused_output).abs().max()
print("Maximum fusion difference:", max_diff.item())

from torch.nn.utils import fuse_conv_bn_eval as torch_fuse_conv_bn_eval

torch_fused_conv = torch_fuse_conv_bn_eval(conv, batch_norm)

with  torch.inference_mode():
    torch_fused_output = torch_fused_conv(x)

max_diff = (fused_output - torch_fused_output).abs().max()
print("Difference  from  PyTorch  fusion:", max_diff.item())




Maximum fusion difference: 9.5367431640625e-07
Difference  from  PyTorch  fusion: 0.0


In [17]:
"""
BatchNorm  通常放在网络的什么位置
    卷积先产生每个通道的线性响应,BatchNorm 调整这些响应的尺度,激活函数再增加非线性
"""

block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, bias=False),
    nn.BatchNorm2d(16),
    nn.ReLU(),
)

x = torch.randn(4, 3, 32, 32)
output = block(x)
print("Output  shape:", output.shape)

Output  shape: torch.Size([4, 16, 32, 32])
